# `gate1_hard_threat/gate.py` — Playground

Manual verification notebook for Gate 1: Hard Threat Screen.

| Function | Status | Notes |
|---|---|---|
| `get_shared_market_data()` | ✅ built | Call once per batch — VIX, SPY, macro are market-wide |
| `screen_gate1_hard_threats(candidate, shared, portfolio_value, daily_pnl)` | ✅ built | 8 rule-based checks, zero Claude cost |

**Block thresholds (from `constants.BLOCK_THRESHOLDS`):**

| Check | Threshold | Block condition |
|---|---|---|
| loss_limit | 3% | portfolio down ≥ 3% today |
| vix | 30.0 | VIX level ≥ 30 |
| spy | -1.5% | SPY down > 1.5% on the day |
| sector | -2.0% | Sector ETF down > 2% on the day |
| premarket_gap | ±3% | \|gap\| ≥ 3% before open |
| macro_event | 2 hours | High-impact event within 2 hours |
| earnings | — | Reports today (any hour) or tomorrow bmo |
| filings_8k | — | Any 8-K filed in the last 24h |

In [1]:
import sys
import pathlib

# Add 02_intelligence/ (for constants + helpers) and gate1_hard_threat/ (for gate.py directly)
# Importing gate.py directly by folder — same pattern as helpers/*_playground.ipynb notebooks.
intelligence_dir = pathlib.Path('backend/02_intelligence').resolve()
gate1_dir        = intelligence_dir / 'gate1_hard_threat'

for p in [str(intelligence_dir), str(gate1_dir)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from gate import get_shared_market_data, screen_gate1_hard_threats

---
## `get_shared_market_data()`

**Expected output:** `{vix: dict|None, spy: dict|None, macro_hours: float|None}`  
**Usage pattern:** Call once before the candidate loop. Pass result to every `screen_gate1_hard_threats()` call.  
**Returns None only if:** VIX, SPY, and macro all fail simultaneously (very unlikely).

In [2]:
# Happy path — live market data
shared = get_shared_market_data()
print(shared)

{'vix': {'level': 15.32, 'change_pct_today': -0.0267, 'prior_close': 15.74}, 'spy': {'price': 756.48, 'change_pct_today': 0.0025, 'prior_close': 754.6}, 'macro_hours': 149.9}


In [3]:
# Inspect individual keys
if shared:
    vix = shared['vix']
    spy = shared['spy']
    macro = shared['macro_hours']

    print(f'VIX level:         {vix["level"] if vix else "N/A"}')
    print(f'VIX change today:  {vix["change_pct_today"]:+.2%}' if vix else 'VIX: fetch failed')
    print()
    print(f'SPY price:         {spy["price"] if spy else "N/A"}')
    print(f'SPY change today:  {spy["change_pct_today"]:+.2%}' if spy else 'SPY: fetch failed')
    print()
    if macro is not None:
        print(f'Next macro event:  {macro}h away')
    else:
        print('Next macro event:  None (no high-impact event in 7 days)')
else:
    print('shared returned None — all market fetches failed')

VIX level:         15.32
VIX change today:  -2.67%

SPY price:         756.48
SPY change today:  +0.25%

Next macro event:  149.9h away


---
## `screen_gate1_hard_threats()` — Happy path

Normal trading day, small portfolio loss. All 8 checks should pass.

In [4]:
candidate_aapl = {
    'ticker': 'AAPL',
    'sector': 'Electronic Technology',
}

result = screen_gate1_hard_threats(
    candidate=candidate_aapl,
    shared=shared,
    portfolio_value=100_000,
    daily_pnl=-500,   # -0.5%, well within 3% limit
)

print(f'passed:       {result["passed"]}')
print(f'block_reason: {result["block_reason"]}')
print()
for name, check in result['checks'].items():
    status = '✅ PASS' if check['passed'] else '❌ BLOCK'
    print(f'  {status}  {name}')

[gate1] AAPL: passed all 8 checks
passed:       True
block_reason: None

  ✅ PASS  loss_limit
  ✅ PASS  vix
  ✅ PASS  spy
  ✅ PASS  sector
  ✅ PASS  premarket_gap
  ✅ PASS  macro_event
  ✅ PASS  earnings
  ✅ PASS  filings_8k


---
## Forced block — `loss_limit`

Portfolio lost -5% today. Should block immediately on check 1, no network needed.

In [8]:
blocked = screen_gate1_hard_threats(
    candidate=candidate_aapl,
    shared=shared,
    portfolio_value=100_000,
    daily_pnl=-5_000,   # -5%, exceeds 3% limit
)

print(f'passed:       {blocked["passed"]}')
print(f'block_reason: {blocked["block_reason"]}')
print()
print(f'loss_pct: {blocked["checks"]["loss_limit"]["loss_pct"]:+.2%}')
assert blocked['block_reason'] == 'loss_limit'
print('loss_limit block correctly triggered ✅')

[gate1] AAPL: BLOCKED — loss_limit
passed:       False
block_reason: loss_limit

loss_pct: -5.00%
loss_limit block correctly triggered ✅


---
## Forced block — `vix` (mocked shared data)

Simulate a high-fear market day by overriding the shared dict with VIX = 35.

In [9]:
# Simulate VIX = 35 (above 30 threshold)
shared_high_vix = {
    'vix': {'level': 35.0, 'change_pct_today': 0.18, 'prior_close': 29.7},
    'spy': shared['spy'] if shared else None,
    'macro_hours': shared['macro_hours'] if shared else None,
}

result_vix = screen_gate1_hard_threats(
    candidate=candidate_aapl,
    shared=shared_high_vix,
    portfolio_value=100_000,
    daily_pnl=0,
)

print(f'passed:       {result_vix["passed"]}')
print(f'block_reason: {result_vix["block_reason"]}')
assert result_vix['block_reason'] == 'vix'
print('VIX block correctly triggered ✅')

[gate1] AAPL: BLOCKED — vix
passed:       False
block_reason: vix
VIX block correctly triggered ✅


---
## Forced block — `spy` (mocked shared data)

Simulate SPY down -2% — exceeds the -1.5% threshold.

In [10]:
shared_spy_down = {
    'vix': {'level': 22.0, 'change_pct_today': 0.05, 'prior_close': 21.0},  # VIX OK
    'spy': {'price': 490.0, 'change_pct_today': -0.02, 'prior_close': 500.0},  # SPY -2%
    'macro_hours': shared['macro_hours'] if shared else None,
}

result_spy = screen_gate1_hard_threats(
    candidate=candidate_aapl,
    shared=shared_spy_down,
    portfolio_value=100_000,
    daily_pnl=0,
)

print(f'passed:       {result_spy["passed"]}')
print(f'block_reason: {result_spy["block_reason"]}')
assert result_spy['block_reason'] == 'spy'
print('SPY block correctly triggered ✅')

[gate1] AAPL: BLOCKED — spy
passed:       False
block_reason: spy
SPY block correctly triggered ✅


---
## Forced block — `macro_event` (mocked shared data)

Simulate a FOMC announcement in 1.5 hours — under the 2-hour threshold.

In [11]:
shared_macro_imminent = {
    'vix': shared['vix'] if shared else None,
    'spy': shared['spy'] if shared else None,
    'macro_hours': 1.5,  # FOMC in 90 minutes
}

result_macro = screen_gate1_hard_threats(
    candidate=candidate_aapl,
    shared=shared_macro_imminent,
    portfolio_value=100_000,
    daily_pnl=0,
)

print(f'passed:       {result_macro["passed"]}')
print(f'block_reason: {result_macro["block_reason"]}')
assert result_macro['block_reason'] == 'macro_event'
print('macro_event block correctly triggered ✅')

[gate1] AAPL: BLOCKED — macro_event
passed:       False
block_reason: macro_event
macro_event block correctly triggered ✅


---
## Variations — multiple candidates, one shared fetch

The real pipeline pattern: fetch shared once, loop through candidates.

In [12]:
candidates = [
    {'ticker': 'AAPL', 'sector': 'Electronic Technology'},
    {'ticker': 'NVDA', 'sector': 'Electronic Technology'},
    {'ticker': 'BAC',  'sector': 'Finance'},
    {'ticker': 'T',    'sector': 'Communications'},
]

# shared already fetched above — reuse it
for c in candidates:
    r = screen_gate1_hard_threats(c, shared, portfolio_value=100_000, daily_pnl=-200)
    status = 'PASS' if r['passed'] else f'BLOCK ({r["block_reason"]})'
    print(f'  {c["ticker"]:<6}  {status}')

[gate1] AAPL: passed all 8 checks
  AAPL    PASS
[gate1] NVDA: passed all 8 checks
  NVDA    PASS
[gate1] BAC: passed all 8 checks
  BAC     PASS
[gate1] T: passed all 8 checks
  T       PASS


---
## Inspect raw checks dict

Full detail view of all 8 checks for one candidate.

In [13]:
import json

result_detail = screen_gate1_hard_threats(
    candidate={'ticker': 'NVDA', 'sector': 'Electronic Technology'},
    shared=shared,
    portfolio_value=100_000,
    daily_pnl=-800,
)

print(json.dumps(result_detail['checks'], indent=2, default=str))

[gate1] NVDA: passed all 8 checks
{
  "loss_limit": {
    "passed": true,
    "breached": false,
    "loss_pct": -0.008
  },
  "vix": {
    "passed": true,
    "level": 16.7,
    "change_pct_today": -0.0036,
    "prior_close": 16.76
  },
  "spy": {
    "passed": true,
    "price": 745.64,
    "change_pct_today": 0.0039,
    "prior_close": 742.72
  },
  "sector": {
    "passed": true,
    "etf_ticker": "XLK",
    "change_pct_today": 0.01,
    "prior_close": 178.6
  },
  "premarket_gap": {
    "passed": true,
    "gap_pct": 0.0261,
    "direction": "up",
    "prior_close": 215.33,
    "premarket_price": 220.96
  },
  "macro_event": {
    "passed": true,
    "hours_to_next_macro": 124.7
  },
  "earnings": {
    "passed": true,
    "reports_today": false,
    "reports_tomorrow": false,
    "hour": null,
    "report_date": null
  },
  "filings_8k": {
    "passed": true,
    "filing_count": 0,
    "filings": []
  }
}


---
## Free play

Try any ticker or mocked scenario.

In [ ]:
# Your experiment here
